# 🔴 THOMASSON ENGINE v7.0 — RC Lens Data Scouting
## Budget Edition | Données Enrichies FBRef + SofaScore | RC Lens Squad Exclus

**Objectif** : Identifier scientifiquement le successeur d'Adrien Thomasson (MF Box-to-Box, RC Lens)  
parmi les milieux de terrain des Big 5 européens, saison 2024/25.

**Nouveautés v7** :
- ✅ Exclusion stricte des joueurs RC Lens du pool de candidats
- ✅ Fusion de 4 datasets (SofaScore + FBRef : Shooting, Miscellaneous, Defensive)
- ✅ Données budgétaires Transfermarkt vérifiées (mars 2026)
- ✅ Score enrichi : Tacles gagnés (TklW), Centres/90 intégrés
- ✅ Budget Lens ~8-10M€ comme filtre opérationnel

**Sources** : SofaScore Big 5 2024/25 · FBRef Shooting/Misc/Defensive · Transfermarkt  
**Auteurs** : Antoine Michel & Julien Verdier (RC Lens Cellule Data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# ─── Palette RC Lens ──────────────────────────────────────────────────────────
LENS_ROUGE = '#E30613'
LENS_OR    = '#F5A623'
LENS_FOND  = '#0F0F0F'
LENS_BLANC = '#F2F2F2'
LENS_GRIS  = '#888888'
ACCENT     = '#00A8E8'
VERT       = '#4CAF50'
ORANGE     = '#FF9800'
PURPLE     = '#9C27B0'

plt.rcParams.update({
    'figure.facecolor': LENS_FOND, 'axes.facecolor': LENS_FOND,
    'text.color': LENS_BLANC, 'axes.labelcolor': LENS_BLANC,
    'xtick.color': LENS_GRIS, 'ytick.color': LENS_GRIS,
    'axes.edgecolor': '#2A2A2A', 'axes.grid': True,
    'grid.color': '#1F1F1F', 'grid.linewidth': 0.6
})

print("✅ Imports OK | Palette RC Lens initialisée")


## 1. Chargement & Fusion des Datasets

4 fichiers CSV sont utilisés :
- `big5_stats.csv` : stats SofaScore (rating, passes clés, xA, tacles...)
- `Shooting_Stats.csv` : FBRef — tirs, SoT/90 (FBRef)
- `Miscellaneous_Stats.csv` : FBRef — centres totaux, tacles gagnés
- `stats_joueurs_final.csv` : FBRef Defensive — tacles gagnés détaillés, blocs, interceptions


In [ ]:
# ─── Chargement des 4 datasets ─────────────────────────────────────────────
big5   = pd.read_csv('big5_stats.csv', sep=';', on_bad_lines='skip')
misc   = pd.read_csv('Miscellaneous_Stats.csv', on_bad_lines='skip')
shoot  = pd.read_csv('Shooting_Stats.csv', on_bad_lines='skip')
defdf  = pd.read_csv('stats_joueurs_final.csv', on_bad_lines='skip')

# Normalisation clé de jointure
for df in [misc, shoot]:
    df['_player'] = df['1_level_0_Player'].astype(str).str.strip()
defdf['_player'] = defdf['Player'].astype(str).str.strip()

# Colonnes numériques big5
COLS_NUM = [
    'minutesPlayed','rating','goals','assists','expectedGoals','expectedAssists',
    'keyPasses','tackles','interceptions','accuratePassesPercentage',
    'successfulDribbles','bigChancesCreated','fouls','wasFouled',
    'groundDuelsWonPercentage','aerialDuelsWonPercentage','player_age',
    'appearances','accurateFinalThirdPasses','clearances','accurateCrosses'
]
for c in COLS_NUM:
    big5[c] = pd.to_numeric(big5[c], errors='coerce')

# Déduplication + filtre 600 min
big5 = big5.dropna(subset=['player__name','minutesPlayed','rating'])
big5 = big5.sort_values('minutesPlayed', ascending=False).drop_duplicates('player__name', keep='first')
big5 = big5[big5['minutesPlayed'] >= 600].copy()
big5['_player'] = big5['player__name'].str.strip()

# ─── Extraction colonnes supplémentaires ─────────────────────────────────────
shoot_s = shoot[['_player','Standard_SoT/90','Standard_Sh/90']].copy()
shoot_s.columns = ['_player','SoT_p90','Sh_p90']
for c in ['SoT_p90','Sh_p90']:
    shoot_s[c] = pd.to_numeric(shoot_s[c], errors='coerce')

misc_s = misc[['_player','Performance_Crs','Performance_TklW']].copy()
misc_s.columns = ['_player','Crosses_total','TklW_misc']
for c in ['Crosses_total','TklW_misc']:
    misc_s[c] = pd.to_numeric(misc_s[c], errors='coerce')

def_s = defdf[['_player','Tkl','TklW','Blocks','Int','Tkl+Int','Clr']].copy()
def_s.columns = ['_player','Tkl_tot','TklW_def','Blocks_def','Int_def','TklInt','Clr_def']
for c in def_s.columns[1:]:
    def_s[c] = pd.to_numeric(def_s[c], errors='coerce')

# ─── Fusion ──────────────────────────────────────────────────────────────────
df = (big5
      .merge(shoot_s, on='_player', how='left')
      .merge(misc_s,  on='_player', how='left')
      .merge(def_s,   on='_player', how='left'))

print(f"Dataset fusionné : {df.shape[0]} joueurs × {df.shape[1]} colonnes")


## 2. Exclusion RC Lens Squad

Tous les joueurs actuellement sous contrat avec le RC Lens sont exclus du pool de candidats.
L'exclusion se fait sur le nom d'équipe `team__name` (méthode la plus robuste).

**Joueurs RC Lens exclus** : Thomasson (référence uniquement), Sangaré, Udol, Haidara, Thauvin, 
Édouard, Risser, Baidoo, Guilavogui, Abdulhamid, Sima, Bermont, Sotoca, Aguilar, Masuaku, etc.


In [ ]:
# ─── Exclusion joueurs RC Lens par nom d'équipe ─────────────────────────────
lens_mask = df['team__name'].str.contains('Lens', na=False, case=False)

# Afficher les joueurs exclus
print("Joueurs RC Lens exclus du pool candidats :")
print(df[lens_mask]['player__name'].tolist())

# Thomasson conservé séparément comme référence
T_raw = df[df['player__name'].str.contains('Thomasson', na=False)].copy()
df_clean = df[~lens_mask].copy()

print(f"\nPool après exclusion RC Lens : {df_clean.shape[0]} joueurs")


## 3. Calcul des Features /90 min

In [ ]:
def add_p90_features(d):
    d = d.copy()
    d['p90'] = d['minutesPlayed'] / 90
    p90_cols = [
        'goals','assists','expectedGoals','expectedAssists','keyPasses','tackles',
        'interceptions','bigChancesCreated','successfulDribbles','fouls','wasFouled',
        'accurateFinalThirdPasses','clearances','accurateCrosses'
    ]
    for c in p90_cols:
        d[f'{c}_p90'] = d[c] / d['p90']
    # Données FBRef /90
    d['TklW_p90']    = d['TklW_def'] / d['p90']   # Tacles gagnés /90
    d['Crosses_p90'] = d['Crosses_total'] / d['p90']  # Centres /90
    return d

df_clean = add_p90_features(df_clean)
T_df     = add_p90_features(T_raw)

print("✅ Features /90 calculées")
print(f"Thomasson — Tacles/90: {T_df['tackles_p90'].iloc[0]:.2f} | TklW/90: {T_df['TklW_p90'].iloc[0]:.2f} | Passes Clés/90: {T_df['keyPasses_p90'].iloc[0]:.2f}")


## 4. Modèle de Scoring Composite Enrichi

### Formule Thomasson Engine v7

```
Score Final = Pressing (45%) + Création (45%) + Circulation (10%)
```

**Pressing (45%)** — enrichi avec TklW (tacles gagnés) depuis FBRef :
- Tacles/90 : 18% | Tacles Gagnés/90 (FBRef) : 14% | Interceptions/90 : 18%
- Duels sol (%) : 15% | Dégagements/90 : 10% | Fautes provoquées/90 : 10% | Rating SofaScore : 15%

**Création (45%)** — enrichi avec Centres/90 :
- Passes Clés/90 : 25% | Big Chances Créées/90 : 20% | Passes Décisives/90 : 20%
- xA/90 : 15% | Dribbles Réussis/90 : 10% | Centres/90 (FBRef) : 10%

**Circulation (10%)** : Précision passes (%)


In [ ]:
def compute_scores(d):
    d = d.copy()
    
    # PRESSING (45%) — enrichi FBRef TklW
    d['score_pressing'] = (
        d['tackles_p90'].fillna(0)              * 0.18 +
        d['TklW_p90'].fillna(0)                 * 0.14 +
        d['interceptions_p90'].fillna(0)         * 0.18 +
        d['groundDuelsWonPercentage'].fillna(0)/100 * 0.15 +
        d['clearances_p90'].fillna(0)            * 0.10 +
        d['fouls_p90'].fillna(0)                 * 0.10 +
        d['rating'].fillna(0)                    * 0.15
    )
    
    # CRÉATION (45%) — enrichi centres/90
    d['score_creation'] = (
        d['keyPasses_p90'].fillna(0)             * 0.25 +
        d['bigChancesCreated_p90'].fillna(0)     * 0.20 +
        d['assists_p90'].fillna(0)               * 0.20 +
        d['expectedAssists_p90'].fillna(0)       * 0.15 +
        d['successfulDribbles_p90'].fillna(0)    * 0.10 +
        d['Crosses_p90'].fillna(0)               * 0.10
    )
    
    # SCORE FINAL
    d['score_final'] = (
        d['score_pressing'] * 0.45 +
        d['score_creation'] * 0.45 +
        d['accuratePassesPercentage'].fillna(0)/100 * 0.10
    )
    return d

df_clean = compute_scores(df_clean)
T_df     = compute_scores(T_df)

# Normalisation 0-100 sur le pool complet (avec Thomasson inclus pour référence)
mf_all = pd.concat([
    df_clean[df_clean['player_position'].str.contains('MF', na=False)],
    T_df
], ignore_index=True)

for s in ['score_final','score_pressing','score_creation']:
    mf_all[f'{s}_norm'] = MinMaxScaler((0,100)).fit_transform(mf_all[[s]])

T = mf_all[mf_all['player__name'].str.contains('Thomasson', na=False)].iloc[0]
mf = mf_all[~mf_all['player__name'].str.contains('Thomasson', na=False)].copy()

print(f"Milieux dans le pool (RC Lens exclu) : {len(mf)}")
print(f"\nThomasson (référence) :")
print(f"  Score Final : {T['score_final_norm']:.1f}/100")
print(f"  Pressing    : {T['score_pressing_norm']:.1f}/100")
print(f"  Création    : {T['score_creation_norm']:.1f}/100")
print(f"  Tacles/90   : {T['tackles_p90']:.2f} | TklW/90 : {T['TklW_p90']:.2f}")
print(f"  Passes Clés/90 : {T['keyPasses_p90']:.2f} | xA/90 : {T['expectedAssists_p90']:.3f}")
print(f"  Précision passes : {T['accuratePassesPercentage']:.1f}% | Rating : {T['rating']:.2f}")


## 5. Top Candidats & Filtre Budget

In [ ]:
# Contexte budgétaire RC Lens (Source : Transfermarkt, mars 2026)
MARKET_VALUES = {
    'Sean Longstaff' : 8.0,   # Leeds Utd, Championship → accessible
    'Gvidas Gineitis': 7.0,   # Torino, Serie A mid-table
    'Kévin Danois'   : 10.0,  # Auxerre, Ligue 1
    'Azzedine Ounahi': 10.0,  # Girona, La Liga
    'Krépin Diatta'  : 12.0,  # AS Monaco, Ligue 1 (stretch)
    'Pablo Fornals'  : 8.0,   # Real Betis, La Liga
}
BUDGET_MAX = 10.0  # Budget max habituel RC Lens
BUDGET_STRETCH = 12.0  # Exception acceptable

# Top 20 candidats ≤28 ans
top20 = mf[mf['player_age'] <= 28].nlargest(20, 'score_final_norm')
print("=== TOP 20 MILIEUX ≤28 ANS (RC LENS EXCLU) ===")
print(f"{'Joueur':<28} {'Club':<24} {'Ligue':<16} {'Age':>3} {'Score':>6} {'Press':>6} {'Crea':>6}")
print("-"*95)
for _,r in top20.iterrows():
    print(f"{r['player__name']:<28} {r['team__name']:<24} {r['league_name']:<16} {r['player_age']:>3.0f} {r['score_final_norm']:>6.1f} {r['score_pressing_norm']:>6.1f} {r['score_creation_norm']:>6.1f}")

print("\n=== SHORTLIST BUDGET (candidats accessibles ≤12M€) ===")
print(f"{'Joueur':<26} {'Score':>6} {'Press':>6} {'Créa':>6} {'KP/90':>6} {'Tack/90':>7} {'TklW/90':>7} {'Int/90':>7} {'Budget':>8}")
print("-"*90)
for name, budget in MARKET_VALUES.items():
    r = mf[mf['player__name'].str.contains(name.split()[-1], na=False, case=False)]
    if len(r):
        r = r.iloc[0]
        status = "OK" if budget <= BUDGET_MAX else "STRETCH" if budget <= BUDGET_STRETCH else "NON"
        print(f"{r['player__name']:<26} {r['score_final_norm']:>6.1f} {r['score_pressing_norm']:>6.1f} {r['score_creation_norm']:>6.1f} {r['keyPasses_p90']:>6.2f} {r['tackles_p90']:>7.2f} {r['TklW_p90']:>7.2f} {r['interceptions_p90']:>7.2f} {budget:>6.1f}M€ [{status}]")


## 6. Visualisations

In [ ]:
# ─── Radar Thomasson ─────────────────────────────────────────────────────────
CATS = ['Passes Clés\n(/90)','Tacles\n(/90)','Intercept.\n(/90)','xA\n(/90)',
        'Big Chances\n(/90)','Dribbles\n(/90)','Précision\nPasses','Duels Sol\n(%)']
N = len(CATS)
mv = [mf['keyPasses_p90'].quantile(.95), mf['tackles_p90'].quantile(.95),
      mf['interceptions_p90'].quantile(.95), mf['expectedAssists_p90'].quantile(.95),
      mf['bigChancesCreated_p90'].quantile(.95), mf['successfulDribbles_p90'].quantile(.95), 1., 1.]
angles = [n/float(N)*2*np.pi for n in range(N)] + [0]

def get_r(row):
    raw = [row['keyPasses_p90'], row['tackles_p90'], row['interceptions_p90'],
           row['expectedAssists_p90'], row['bigChancesCreated_p90'],
           row['successfulDribbles_p90'], row['accuratePassesPercentage']/100,
           row['groundDuelsWonPercentage']/100]
    return [min((v or 0)/m, 1.0) for v,m in zip(raw, mv)]

tv = get_r(T) + [get_r(T)[0]]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True), facecolor=LENS_FOND)
fig.patch.set_facecolor(LENS_FOND); ax.set_facecolor(LENS_FOND)
for r in [.2,.4,.6,.8,1.]:
    ax.plot(angles, [r]*len(angles), '-', color='#1F1F1F', lw=.8)
ax.fill(angles, tv, color=LENS_ROUGE, alpha=.25)
ax.plot(angles, tv, 'o-', color=LENS_ROUGE, lw=2.5, ms=7)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(CATS, size=9.5, color=LENS_BLANC)
ax.set_yticks([]); ax.spines['polar'].set_color('#222')
ax.set_title(f"ADRIEN THOMASSON — Empreinte Tactique\nScore : {T['score_final_norm']:.1f}/100 | Pressing : {T['score_pressing_norm']:.1f} | Création : {T['score_creation_norm']:.1f}",
             color=LENS_BLANC, fontsize=13, fontweight='bold', pad=28)
plt.tight_layout(); plt.savefig('v7_radar_thomasson.png', dpi=130, bbox_inches='tight', facecolor=LENS_FOND)
plt.show(); print("Radar Thomasson généré")


In [ ]:
# ─── Scatter Pressing vs Création ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 8), facecolor=LENS_FOND)
mf_plot = mf[mf['player_age'] <= 30]
ax.scatter(mf_plot['score_pressing_norm'], mf_plot['score_creation_norm'],
           c='#1A1A1A', s=18, alpha=0.5, zorder=1)
ax.axvspan(40,80, alpha=0.04, color=LENS_OR)
ax.axhspan(35,70, alpha=0.04, color=LENS_OR)
ax.text(42, 67, 'ZONE HYBRIDE', color=LENS_OR, fontsize=9, alpha=0.7)

for name, col in [('Longstaff',LENS_OR),('Gineitis',VERT),('Danois',ACCENT),
                   ('Ounahi',ORANGE),('Diatta',PURPLE),('Fornals','#AAAAAA')]:
    r = mf[mf['player__name'].str.contains(name, na=False, case=False)]
    if len(r):
        r = r.iloc[0]
        ax.scatter(r['score_pressing_norm'], r['score_creation_norm'],
                   c=[col], s=220, zorder=5, edgecolors=LENS_BLANC, linewidths=1.2)
        ax.annotate(r['player__name'].split()[-1],
                    xy=(r['score_pressing_norm'], r['score_creation_norm']),
                    xytext=(5,5), textcoords='offset points', fontsize=8.5, color=col, fontweight='bold')

ax.scatter(T['score_pressing_norm'], T['score_creation_norm'],
           c=LENS_ROUGE, s=380, zorder=6, marker='*', edgecolors=LENS_BLANC, linewidths=1.5)
ax.annotate('THOMASSON', xy=(T['score_pressing_norm'], T['score_creation_norm']),
            xytext=(6,-16), textcoords='offset points', fontsize=10, color=LENS_ROUGE, fontweight='bold')

ax.set_xlabel('Score Pressing (0-100)', fontsize=12)
ax.set_ylabel('Score Création (0-100)', fontsize=12)
ax.set_title('SCATTER — Pressing vs Création | 1038 MF Big 5 | RC Lens exclu',
             color=LENS_BLANC, fontsize=13, fontweight='bold', pad=12)
plt.tight_layout(); plt.savefig('v7_scatter.png', dpi=130, bbox_inches='tight', facecolor=LENS_FOND)
plt.show()


In [ ]:
# ─── Radars comparatifs Top 3 ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18,7), subplot_kw=dict(polar=True), facecolor=LENS_FOND)
fig.patch.set_facecolor(LENS_FOND)

for ax, (kw, col, title) in zip(axes, [
    ('Longstaff', LENS_OR,  'SEAN LONGSTAFF\nLeeds | 28 ans | ~8M€'),
    ('Gineitis',  VERT,     'GVIDAS GINEITIS\nTorino | 21 ans | ~7M€'),
    ('Danois',    ACCENT,   'KÉVIN DANOIS\nAuxerre | 21 ans | ~10M€'),
]):
    r = mf[mf['player__name'].str.contains(kw, na=False, case=False)]
    if not len(r): continue
    r = r.iloc[0]; cv = get_r(r) + [get_r(r)[0]]
    ax.set_facecolor(LENS_FOND)
    for ring in [.2,.4,.6,.8,1.]:
        ax.plot(angles, [ring]*len(angles), '-', color='#1F1F1F', lw=.8)
    ax.fill(angles, tv, color=LENS_ROUGE, alpha=.15)
    ax.plot(angles, tv, 'o-', color=LENS_ROUGE, lw=2, ms=5, label='Thomasson')
    ax.fill(angles, cv, color=col, alpha=.2)
    ax.plot(angles, cv, 's-', color=col, lw=2.5, ms=6, label=r['player__name'])
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(CATS, size=7.5, color=LENS_BLANC)
    ax.set_yticks([]); ax.spines['polar'].set_color('#222')
    sim = (1 - np.mean(np.abs(np.array(tv[:-1]) - np.array(cv[:-1])))) * 100
    ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.12), fontsize=8.5, framealpha=.3, labelcolor=LENS_BLANC)
    ax.set_title(f"{title}\nSimilarité : {sim:.0f}%", color=LENS_BLANC, fontsize=10, fontweight='bold', pad=22)

fig.suptitle("RADARS TOP 3 vs THOMASSON | Données enrichies SofaScore + FBRef",
             color=LENS_BLANC, fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.savefig('v7_radars.png', dpi=130, bbox_inches='tight', facecolor=LENS_FOND)
plt.show()


In [ ]:
# ─── Heatmap Percentiles ─────────────────────────────────────────────────────
METRICS_H = {
    'Rating SofaScore'   : 'rating',
    'Tacles/90'          : 'tackles_p90',
    'Tacles Gagnés/90'   : 'TklW_p90',
    'Interceptions/90'   : 'interceptions_p90',
    'Passes Clés/90'     : 'keyPasses_p90',
    'xA/90'              : 'expectedAssists_p90',
    'Big Chances/90'     : 'bigChancesCreated_p90',
    'Dribbles/90'        : 'successfulDribbles_p90',
    'Centres/90'         : 'Crosses_p90',
    'Précision Passes'   : 'accuratePassesPercentage',
    'Duels Sol (%)'      : 'groundDuelsWonPercentage',
}

shortlist = ['Sean Longstaff','Gvidas Gineitis','Kévin Danois','Azzedine Ounahi','Krépin Diatta']
rows_list = [T.to_frame().T.copy()]
found_names = ['Thomasson\n(Réf.)']
for n in shortlist:
    r = mf[mf['player__name'].str.contains(n.split()[-1], na=False, case=False)]
    if len(r):
        rows_list.append(r.iloc[0:1].copy())
        found_names.append(n.split()[-1])

all_p = pd.concat(rows_list, ignore_index=True)
mlabels = list(METRICS_H.keys())
mcols   = list(METRICS_H.values())
matrix  = np.zeros((len(mcols), len(all_p)))

for j in range(len(all_p)):
    for i, col in enumerate(mcols):
        val = float(all_p.iloc[j][col]) if col in all_p.columns and not pd.isna(all_p.iloc[j][col]) else 0
        ref = mf_all[col].dropna() if col in mf_all else pd.Series([0])
        matrix[i,j] = (ref < val).mean() * 100

fig, ax = plt.subplots(figsize=(14, 7), facecolor=LENS_FOND)
cmap = mcolors.LinearSegmentedColormap.from_list('lens', ['#0F0F0F','#1A1A3A','#003399','#CC0000','#FF6600','#FFD700'])
im = ax.imshow(matrix, cmap=cmap, aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(len(found_names))); ax.set_xticklabels(found_names, fontsize=10, color=LENS_BLANC, fontweight='bold')
ax.set_yticks(range(len(mlabels))); ax.set_yticklabels(mlabels, fontsize=9.5, color=LENS_BLANC)
for i in range(len(mlabels)):
    for j in range(len(all_p)):
        c = LENS_BLANC if matrix[i,j] < 75 else '#0A0A0A'
        ax.text(j, i, f'{matrix[i,j]:.0f}', ha='center', va='center', fontsize=9, color=c, fontweight='bold')
plt.Rectangle((-0.5,-0.5),1,len(mlabels),linewidth=2.5,edgecolor=LENS_ROUGE,facecolor='none')
plt.colorbar(im, ax=ax, fraction=.02, pad=.02).set_label('Percentile', color=LENS_BLANC)
ax.set_title("HEATMAP — 11 KPIs | Percentile vs Pool Big 5 | RC Lens exclu",
             color=LENS_BLANC, fontsize=13, fontweight='bold', pad=14)
plt.tight_layout(); plt.savefig('v7_heatmap.png', dpi=130, bbox_inches='tight', facecolor=LENS_FOND)
plt.show()


## 7. Conclusion & Recommandation Finale

### Budget RC Lens (Source : Transfermarkt, saison 25/26)
- **Dépenses 25/26** : 56.4M€ (dont Sangaré 8M€, Baidoo 8M€, Agbonifo 6.5M€, Thauvin 6M€)
- **Revenus 25/26** : 102M€ → Solde +45.6M€
- **Budget d'acquisition habituel** : ~8-10M€ par joueur hors exception
- **Exception historique** : Wahi 30M€ + Diouf 14M€ en 23/24 uniquement

### Shortlist Finale (après exclusion RC Lens + filtrage humain)

| # | Joueur | Club | Âge | Budget | Score | Verdict |
|---|--------|------|-----|--------|-------|---------|
| 🥇1 | **Sean Longstaff** | Leeds United (PL) | 28 | ~8M€ | 71.4/100 | **Priorité absolue** |
| 🥈2 | **Gvidas Gineitis** | Torino (Serie A) | 21 | ~7M€ | 66.0/100 | Pépite Moneyball |
| 🥉3 | **Kévin Danois** | Auxerre (L1) | 21 | ~10M€ | 53.4* | Clone L1 sous-évalué |
| ⚠️4 | Azzedine Ounahi | Girona (La Liga) | 25 | ~10M€ | 51.7/100 | ⚠️ Risque médical |
| 5 | Krépin Diatta | AS Monaco (L1) | 26 | ~12M€ | 64.6/100 | Stretch budget |

*Score Danois sous-estimé : contexte Auxerre (maintien) déprime ses stats créatives.

### Recommandation #1 : Sean Longstaff
- Profil le plus complet dans le budget | 3.48 tack/90 + 2.07 passes clés/90 + 2.16 TklW/90
- Leeds en Championship → transfert réaliste ~8M€
- Expérience PL disponible pour jouer l'Europe avec Lens


In [ ]:
# Résumé final des stats candidats
print("="*80)
print("THOMASSON ENGINE v7.0 — RÉSUMÉ FINAL | RC LENS 2025/26")
print("="*80)
print(f"\nTHOMASSSON (référence) :")
print(f"  Score: {T['score_final_norm']:.1f} | Pressing: {T['score_pressing_norm']:.1f} | Création: {T['score_creation_norm']:.1f}")
print(f"  Tacks/90: {T['tackles_p90']:.2f} | TklW/90: {T['TklW_p90']:.2f} | KP/90: {T['keyPasses_p90']:.2f} | Rating: {T['rating']:.2f}")

shortlist_final = [
    ('Sean Longstaff',  LENS_OR,  '~8M€',  '#1 PRIORITÉ'),
    ('Gvidas Gineitis', VERT,     '~7M€',  '#2 MONEYBALL'),
    ('Kévin Danois',    ACCENT,   '~10M€', '#3 CLONE L1*'),
    ('Azzedine Ounahi', ORANGE,   '~10M€', '#4 ⚠️MÉDICAL'),
    ('Krépin Diatta',   PURPLE,   '~12M€', '#5 STRETCH'),
]
print()
for name, col, budget, verdict in shortlist_final:
    r = mf[mf['player__name'].str.contains(name.split()[-1], na=False, case=False)]
    if len(r):
        r = r.iloc[0]
        print(f"[{verdict}] {r['player__name']} | {budget}")
        print(f"  Score: {r['score_final_norm']:.1f} | P: {r['score_pressing_norm']:.1f} | C: {r['score_creation_norm']:.1f}")
        print(f"  Tacks/90: {r['tackles_p90']:.2f} | TklW/90: {r['TklW_p90']:.2f} | KP/90: {r['keyPasses_p90']:.2f} | Rating: {r['rating']:.2f}")
        print()
